In [2]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, to_timestamp
from pyspark.sql.window import Window
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable
from helpers import get_table, get_bronze
from functools import reduce
print(pyspark.__version__)

3.5.8


In [3]:
builder = (
    SparkSession.builder
    .appName("silver_processing")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(
    builder,
    extra_packages=["org.postgresql:postgresql:42.7.3"]
).getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

26/04/27 14:56:20 WARN Utils: Your hostname, CVPThuyTTD11-1 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/27 14:56:20 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/thuyttd11/.local/lib/python3.8/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/thuyttd11/.ivy2/cache
The jars for the packages stored in: /home/thuyttd11/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d7d0a8e6-f323-4e81-a01f-fefec768b839;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.3.2 in central
	found io.delta#delta-storage;3.3.2 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.postgresql#postgresql;42.7.3 in central
	found org.checkerframework#checker-qual;3.42.0 in central
:: resolution report :: resolve 343ms :: artifacts dl 12ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.3.2 from central in [default]
	io.delta#delta-storage;3.3.2 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	org.checkerframework#checker-qual;3.42.0 from central in [default]
	org.postgresql#postgresql;42.7.3 from central in [default]
	------------------------

# License allocations silver

Grain: one row per `allocation_id` and `allocation_date`

Processing steps:
+ Read Bronze `license_allocations`.
+ Cast allocation IDs, license IDs, seat numbers, status, timestamps, and lineage fields.
+ Normalize status to a controlled domain.
+ Quarantine null keys, invalid statuses, invalid dates, and invalid seat numbers.
+ Validate `license_id` against Silver license keys and `seat_number <= max_seats`.
+ Deduplicate by `allocation_id`, `allocation_date`, keeping the latest ingest record.
+ Derive active flags, event type, seat change, and allocation date parts.
+ Upsert into Delta path `./silver/license_allocations`.

## Read Bronze license_allocations data

In [30]:
path = "/home/thuyttd11/subscription-analytics/spark-data/bronze/license_allocations"
df_bronze = get_bronze(path, spark=spark)
df_bronze.show(5)

+-------------+----------+-----------+-----------+---------------+--------------------+--------------------+--------------------+--------------------+
|allocation_id|license_id|seat_number|     status|allocation_date|      ingestion_time|            batch_id|         ingest_time|   source_identifier|
+-------------+----------+-----------+-----------+---------------+--------------------+--------------------+--------------------+--------------------+
|            1|         1|          1|deactivated|     2026-10-27|2026-04-21 13:41:...|app-2026042205250...|2026-04-22 12:26:...|jdbc:postgresql:/...|
|            2|         1|          2|     active|     2026-09-19|2026-04-21 13:41:...|app-2026042205250...|2026-04-22 12:26:...|jdbc:postgresql:/...|
|            3|         1|          3|deactivated|     2026-10-10|2026-04-21 13:41:...|app-2026042205250...|2026-04-22 12:26:...|jdbc:postgresql:/...|
|            4|         1|          4|deactivated|     2027-02-04|2026-04-21 13:41:...|app-202

In [31]:
df_bronze.printSchema()
df_bronze.count()

root
 |-- allocation_id: long (nullable = true)
 |-- license_id: long (nullable = true)
 |-- seat_number: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- allocation_date: date (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- ingest_time: timestamp (nullable = true)
 |-- source_identifier: string (nullable = true)



55000

In [32]:
df1 = df_bronze.drop("batch_id", "source_identifier", "ingestion_time") # Need to verify where the ingestion_time comes in

## Cast types

In [33]:
df2 = (
    df1
    .select(
        F.col("allocation_id").cast("bigint").alias("allocation_id"),
        F.col("license_id").cast("bigint").alias("license_id"),
        F.col("seat_number").cast("int").alias("seat_number"),
        F.col("status").cast("string").alias("status"),
        F.col("allocation_date").cast("timestamp").alias("allocation_date"),
        F.col("ingest_time").cast("timestamp").alias("ingest_time")
    )
)
# df2.show(5)

In [34]:
# df2.select("status").distinct().show()

## Validate required columns and domains

In [35]:
valid_allocation_status = ["active", "deactivated"]
df3 = df2.filter(col("status").isin(valid_allocation_status))
df3_quarantine = df2.filter(~col("status").isin(valid_allocation_status))
# df3.count(), df3_quarantine.count()

## Deduplicate at allocation grain

In [36]:
w = Window.partitionBy("allocation_id", "allocation_date").orderBy(F.col("ingest_time").desc())

df_allocations_ranked = df3.withColumn("rn", F.row_number().over(w))

df4 = df_allocations_ranked.filter(F.col("rn") == 1).drop("rn", "validation_error")
df4_quarantine = (
    df_allocations_ranked
    .filter(F.col("rn") > 1)
    .drop("rn")
    .withColumn("validation_error", F.lit("duplicate allocation_id"))
)
# df4.count(), df4_quarantine.count()

## Validate license_id and seat range

In [37]:
silver_license_keys_path = "./silver/licenses"
df_silver_license_keys_ref = (
    spark.read
    .format("delta")
    .load(silver_license_keys_path)
).select("license_id")

df5 = (
    df4
    .join(
        df_silver_license_keys_ref,
        on="license_id",
        how="left_semi"   # existence check only
    )
    .filter(col("seat_number").between(1, 550))
)

df5_quarantine = (
    df4
    .join(
        df_silver_license_keys_ref,
        on="license_id",
        how="left_anti"   # license_id does NOT exist
    )
    .unionByName(
        df4.filter(~col("seat_number").between(1, 550)),
        allowMissingColumns=True
    )
)
# df5.count(), df5_quarantine.count()

## Get all quarantine tables

In [38]:
quarantine_dfs = [
    df3_quarantine,
    df4_quarantine,
    df5_quarantine

]

df_quarantine_all = reduce(
    lambda df1, df2: df1.unionByName(df2, allowMissingColumns=True),
    quarantine_dfs
)

# df_quarantine_all.count()

In [39]:
df_license_allocations_silver = df5

In [40]:
df_license_allocations_silver.columns

['license_id',
 'allocation_id',
 'seat_number',
 'status',
 'allocation_date',
 'ingest_time']

## Upsert strategy

In [41]:
silver_path = "./silver/license_allocations"

silver_cols = df_license_allocations_silver.columns

df_upsert = df_license_allocations_silver.select(*silver_cols)

w = Window.partitionBy("allocation_id", "allocation_date").orderBy(F.col("ingest_time").desc())
df_upsert = (
    df_upsert
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

if DeltaTable.isDeltaTable(spark, silver_path):
    print("Update existing table")
    delta_target = DeltaTable.forPath(spark, silver_path)

    (
        delta_target.alias("target")
        .merge(df_upsert.alias("source"), "target.allocation_id = source.allocation_id")
        .whenMatchedUpdate(set={c: f"source.{c}" for c in silver_cols if c != "allocation_id"})
        .whenNotMatchedInsert(values={c: f"source.{c}" for c in silver_cols})
        .execute()
    )
else:
    print("Delta table does not exist yet")
    (
        df_upsert
        .write
        .format("delta")
        .mode("overwrite")
        .save(silver_path)
    )

Update existing table


In [42]:
df_license_allocations_silver_read = (
    spark.read
    .format("delta")
    .load(silver_path)
)

df_license_allocations_silver_read.show(5)
df_license_allocations_silver_read.count()

+----------+-------------+-----------+-----------+-------------------+--------------------+
|license_id|allocation_id|seat_number|     status|    allocation_date|         ingest_time|
+----------+-------------+-----------+-----------+-------------------+--------------------+
|         1|            2|          2|     active|2026-09-19 00:00:00|2026-04-22 12:26:...|
|         1|            3|          3|deactivated|2026-10-10 00:00:00|2026-04-22 12:26:...|
|         1|            4|          4|deactivated|2027-02-04 00:00:00|2026-04-22 12:26:...|
|         1|            8|          8|deactivated|2027-01-14 00:00:00|2026-04-22 12:26:...|
|         1|           10|         10|     active|2026-10-26 00:00:00|2026-04-22 12:26:...|
+----------+-------------+-----------+-----------+-------------------+--------------------+
only showing top 5 rows



55000

In [43]:
spark.stop()